# Transformer Foundations, Part 2 of 3: Decoder-Only Language Models

> **Where you are:** [Part 1](01-attention-and-transformer-blocks.ipynb) built the Transformer block. This notebook gives that block a causal view of history, trains it with next-token prediction, generates autoregressively, and then opens a real DistilGPT-2 checkpoint.

The setup cells below are an executable recap of definitions already derived in Part 1. They keep this notebook runnable in a fresh kernel without repeating the exploratory proofs and visualizations.


## Executable Recap from Part 1

Run these compact setup cells in a fresh kernel. Read Part 1 for the derivations; here they are dependencies, not new concepts.


In [ ]:
#  Install dependencies (run once)
import subprocess, sys

required = [
    ("numpy",        "numpy"),
    ("matplotlib",   "matplotlib"),
    ("torch",        "torch"),
    ("seaborn",      "seaborn"),
    ("plotly",       "plotly"),
    ("transformers", "transformers"),
]

# Import each required package, installing it via pip only if it's missing
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        print(f"  installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")


In [ ]:
#  Imports
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import seaborn as sns
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

warnings.filterwarnings('ignore')

# Plotly is optional -- fall back to matplotlib-only 3D plots if it isn't installed
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})
print(f"torch   {torch.__version__}")
print(f"numpy   {np.__version__}")


In [ ]:
#  Vocabulary
VOCAB = {
    "<PAD>": 0, "<BOS>": 1, "<EOS>": 2,
    "the": 3, "a": 4, "cat": 5, "dog": 6,
    "mat": 7, "fence": 8, "sat": 9, "ran": 10,
    "jumped": 11, "on": 12, "over": 13, "big": 14,
}

# Invert the vocab so token IDs can be decoded back to words
IDX2WORD = {v: k for k, v in VOCAB.items()}
VOCAB_SIZE = len(VOCAB)

#  3D Semantic Embeddings (Concreteness, Animacy, Dynamism)
E = {
    "<PAD>": [0.00, 0.00, 0.00], "<BOS>": [0.08, 0.08, 0.15], "<EOS>": [0.08, 0.08, 0.15],
    "the":   [0.05, 0.04, 0.08], "a":     [0.05, 0.04, 0.08],
    "cat":   [0.91, 0.94, 0.38], "dog":   [0.88, 0.92, 0.55],
    "mat":   [0.96, 0.04, 0.04], "fence": [0.93, 0.03, 0.03],
    "sat":   [0.34, 0.18, 0.78], "ran":   [0.28, 0.12, 0.96],
    "jumped":[0.30, 0.14, 0.98], "on":    [0.14, 0.04, 0.18],
    "over":  [0.17, 0.04, 0.24], "big":   [0.44, 0.04, 0.09],
}

# Stack each token's embedding vector in vocab-ID order into one matrix
embedding_matrix = torch.tensor(
    [E[IDX2WORD[i]] for i in range(VOCAB_SIZE)], dtype=torch.float32
)

SENTENCE = "the cat sat on the mat"
TOKENS = SENTENCE.split()

# Encode the running example sentence into vocab IDs
TOKEN_IDS = [VOCAB[w] for w in TOKENS]
SEQ_LEN = len(TOKENS)

print(f"Vocab size       : {VOCAB_SIZE}")
print(f"Embedding shape  : {tuple(embedding_matrix.shape)}  (vocab x 3D)")
print(f"Running sentence : {SENTENCE!r}")
print(f"Token IDs        : {TOKEN_IDS}")
print()
print("Embedding matrix -> Concreteness, Animacy, Dynamism:")

# Skip the special tokens (indices 0-2) when printing the embedding table
for word, vec in list(E.items())[3:]:
    print(f"  {word:<10} [{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}]")


In [ ]:
#  Tokeniser
# Convert a raw string into vocab IDs, optionally wrapping with BOS/EOS markers
def encode(text: str, add_bos: bool = False, add_eos: bool = False):
    ids = [VOCAB.get(w, VOCAB["<PAD>"]) for w in text.lower().split()]
    if add_bos:
        ids = [VOCAB["<BOS>"]] + ids
    if add_eos:
        ids = ids + [VOCAB["<EOS>"]]
    return ids


# Convert vocab IDs back into a whitespace-joined string
def decode(ids):
    return " ".join(IDX2WORD.get(i, "<?>") for i in ids)


phrase = "the big cat jumped over the fence"
enc = encode(phrase)
dec = decode(enc)
print(f"Input  : {phrase!r}")
print(f"Encoded: {enc}")
print(f"Decoded: {dec!r}")
print()
print('With BOS/EOS markers:')
enc2 = encode(phrase, add_bos=True, add_eos=True)
print(f"  {enc2}")
print()
print("  -> One word = one token; BPE splits rare words in real models.")


In [ ]:
#  Sinusoidal Positional Encoding
def sinusoidal_pe(seq_len: int, d_model: int) -> torch.Tensor:
    """Classic additive positional encoding (Vaswani et al. 2017).
    Handles both even and odd d_model gracefully.
    """
    pe = np.zeros((seq_len, d_model), dtype=np.float32)
    positions = np.arange(seq_len)[:, None].astype(np.float32)
    dims = np.arange(0, d_model, 2).astype(np.float32)

    # Geometrically decaying frequency per dimension pair
    freqs = 1.0 / (10000 ** (dims / d_model))

    # Even dims get sine, odd dims get cosine, at the same frequency
    pe[:, 0::2] = np.sin(positions * freqs)
    n_cos = pe[:, 1::2].shape[1]
    pe[:, 1::2] = np.cos(positions * freqs[:n_cos])
    return torch.tensor(pe)


D_VIS = 16
pe_matrix = sinusoidal_pe(SEQ_LEN, D_VIS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Heatmap: PE value at every (token position, dimension) pair
ax = axes[0]
im = ax.imshow(pe_matrix.numpy(), aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_xticks(range(D_VIS))
ax.set_xticklabels([f'd{i}' for i in range(D_VIS)], fontsize=8, rotation=45)
ax.set_yticks(range(SEQ_LEN)); ax.set_yticklabels(TOKENS, fontsize=10)
ax.set_title('Sinusoidal PE - our sentence')
ax.set_xlabel('Embedding dimension'); ax.set_ylabel('Token position')
plt.colorbar(im, ax=ax)

# Line plot: a few dimensions over 50 positions, revealing fast vs. slow oscillation
ax2 = axes[1]
pe_long = sinusoidal_pe(50, D_VIS).numpy()
for i in [0, 2, 6, 14]:
    label = f'dim {i} - {"fast" if i < 4 else "slow"}'
    ax2.plot(pe_long[:, i], label=label, lw=1.8)
ax2.set_title('PE signal per dimension over 50 positions')
ax2.set_xlabel('Token position'); ax2.set_ylabel('PE value')
ax2.legend(fontsize=8); ax2.set_ylim(-1.1, 1.1)

plt.suptitle('Sinusoidal Positional Encoding', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Add the 3D positional signal directly onto the token embeddings
emb_vectors = embedding_matrix[TOKEN_IDS]
pe_3d = sinusoidal_pe(SEQ_LEN, 3)
enriched = emb_vectors + pe_3d
print('After adding 3D sinusoidal PE:')
for i, w in enumerate(TOKENS):

    # Format a 3-vector for aligned printing
    def fmt(v_list):
        return f'[{v_list[0]:+.3f}, {v_list[1]:+.3f}, {v_list[2]:+.3f}]'
    print(f'[{i}] {w:<8}  orig={fmt(emb_vectors[i].tolist())}  pe={fmt(pe_3d[i].tolist())}  sum={fmt(enriched[i].tolist())}')


In [ ]:
#  Working model constants
D_WORK = 16    # functional model dimension
NUM_HEADS = 2  # attention heads
D_HEAD = D_WORK // NUM_HEADS   # 8 per head
D_FF = 32      # feed-forward hidden size


# Splits Q/K/V into n_heads parallel attention heads, then concatenates and projects back
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    # Reshape into (B, H, S, d_head), run scaled dot-product attention per head,
    # then merge the heads back together
    def forward(self, x, mask=None):
        B, S, _ = x.shape
        Q_mh = self.W_Q(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)  # (B, H, S, d_head)
        K_mh = self.W_K(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        V_mh = self.W_V(x).reshape(B, S, self.n_heads, self.d_head).transpose(1, 2)
        scores = (Q_mh @ K_mh.transpose(-2, -1)) / math.sqrt(self.d_head)  # (B, H, S, S)
        if mask is not None:
            scores = scores.masked_fill(mask.bool(), float('-inf'))
        attn_w_mh = torch.softmax(scores, dim=-1)   # (B, H, S, S)
        out = attn_w_mh @ V_mh                       # (B, H, S, d_head)
        out = out.transpose(1, 2).reshape(B, S, self.d_model)
        return self.W_O(out), attn_w_mh


#  Demo
torch.manual_seed(42)
mha = MultiHeadAttention(D_WORK, NUM_HEADS)

proj = nn.Linear(D_MODEL, D_WORK, bias=False)

# Project the 3D toy embeddings up to the working 16-dim model space (no gradient needed for this demo)
with torch.no_grad():
    x_work = proj(embs).unsqueeze(0)   # (1, 6, 16)

mha_out, head_weights = mha(x_work)
print(f'MHA output shape: {tuple(mha_out.shape)}   head_weights shape: {tuple(head_weights.shape)}')

# Plot each head's attention pattern in its own panel
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for h in range(NUM_HEADS):
    ax = axes[h]
    w_h = head_weights[0, h].detach().numpy()
    sns.heatmap(w_h, ax=ax, annot=True, fmt='.2f', cmap='Purples',
                xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5, cbar=False)
    ax.set_title(f'Head {h} attention weights')
    ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.tick_params(axis='x', rotation=30)
plt.suptitle('Multi-Head Attention - each head learns a different relationship', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
#  FeedForward + LayerNorm
# Standard transformer feed-forward block: expand 4x, GELU, project back
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


#  Visualise LayerNorm effect
torch.manual_seed(42)
ffn = FeedForward(D_WORK, D_FF)
norm = nn.LayerNorm(D_WORK, eps=1e-5)

x_raw = x_work[0]   # (6, 16)

# Run the FFN then LayerNorm without tracking gradients (visualisation only)
with torch.no_grad():
    x_after = ffn(x_raw)       # (6, 16) raw FFN output
    x_normed = norm(x_after)   # (6, 16) after LayerNorm

# Plot each stage's per-token activation profile side by side
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axes, [x_raw, x_after, x_normed],
                           ['Input to FFN', 'FFN output (raw)', 'After LayerNorm']):
    data_np = data.detach().numpy()
    for j, token in enumerate(TOKENS):
        vals = data_np[j]
        ax.plot(vals, alpha=0.7, label=f'{token}  mu={vals.mean():.2f}, s={vals.std():.2f}')
    ax.set_title(title); ax.set_xlabel('Hidden dimension'); ax.set_ylabel('Activation value')
    ax.legend(fontsize=7); ax.axhline(0, color='black', lw=0.5, ls='--')
plt.suptitle('FFN activations before and after LayerNorm', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('LayerNorm centres and normalises each token slice.')
print('Mean and std across dimensions after LN:')
x_normed_np = x_normed.detach().numpy()
for j, token in enumerate(TOKENS):
    v = x_normed_np[j]
    print(f'  {token:<8}  mean={v.mean():+.4f}  std={v.std():.4f}')


In [ ]:
#  TransformerBlock
# Pre-LN transformer block: LayerNorm -> MHA -> residual, LayerNorm -> FFN -> residual
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model, eps=1e-5)
        self.mha   = MultiHeadAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-5)
        self.ffn   = FeedForward(d_model, d_ff)

    def forward(self, x, mask=None):

        # Pre-LN attention sublayer, then pre-LN feed-forward sublayer, both with residual adds
        mha_out, attn_w_b = self.mha(self.norm1(x), mask=mask)
        x = x + mha_out
        x = x + self.ffn(self.norm2(x))
        return x, attn_w_b


torch.manual_seed(42)
block1 = TransformerBlock(D_WORK, NUM_HEADS, D_FF)
block2 = TransformerBlock(D_WORK, NUM_HEADS, D_FF)

x0 = x_work.clone()   # (1, 6, 16)

# Run two stacked blocks without tracking gradients (inspection only)
with torch.no_grad():
    x1, aw1 = block1(x0)
    x2, aw2 = block2(x1)

print('Input -> Block 1 -> Block 2:')
print(f'  x0: {tuple(x0.shape)}  norm={float(torch.norm(x0)):.3f}')
print(f'  x1: {tuple(x1.shape)}  norm={float(torch.norm(x1)):.3f}')
print(f'  x2: {tuple(x2.shape)}  norm={float(torch.norm(x2)):.3f}')

# Per-token representation norm at each stage of the stack
fig, ax = plt.subplots(figsize=(8, 4))
norms = {
    'Layer 0 (input)': torch.norm(x0[0], dim=-1).detach().numpy(),
    'Layer 1 output':  torch.norm(x1[0], dim=-1).detach().numpy(),
    'Layer 2 output':  torch.norm(x2[0], dim=-1).detach().numpy(),
}
x_pos = np.arange(SEQ_LEN); width = 0.25

# Plot grouped bars comparing token norms across layers
for k, (label, vals) in enumerate(norms.items()):
    ax.bar(x_pos + k*width, vals, width, label=label, alpha=0.85)
ax.set_xticks(x_pos + width); ax.set_xticklabels(TOKENS)
ax.set_ylabel('Representation L2 norm')
ax.set_title('Token representations grow through transformer blocks')
ax.legend(); plt.tight_layout(); plt.show()

---

## Part 8 - Mini Language Model: Training & Inference

We now wire everything together into a **Mini Language Model** - a decoder-only transformer that learns to predict the next token. This is the architecture of GPT, LLaMA, Mistral etc.

**Training task**: given a context window, predict the next token.


In [ ]:
# Minimal state reused by the W_V and scaling comparisons below.
D_MODEL = embedding_matrix.shape[1]
D_HEAD = D_MODEL
_sentence_embeddings = embedding_matrix[TOKEN_IDS]
_attention_scores = _sentence_embeddings @ _sentence_embeddings.T
attn_w = torch.softmax(_attention_scores / math.sqrt(D_HEAD), dim=-1)
causal_mask = torch.triu(torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool), diagonal=1)
print(f"Recap ready: d_model={D_MODEL}, toy width={D_WORK}, heads={NUM_HEADS}")


> **PyTorch → Keras:** `class MiniLM(nn.Module)` uses `nn.Embedding(vocab_size, d_model)` for token lookup, `self.register_buffer('pe', pe)` to store the (non-trainable) positional-encoding table as part of the module's state, and ties weights by reusing `self.token_emb.weight.T` as the output projection instead of a separate linear layer. **Keras/TF equivalent:** `tf.keras.layers.Embedding(vocab_size, d_model)`; a registered buffer has no single direct Keras equivalent — the closest pattern is storing it as a plain (non-`tf.Variable`) attribute or a non-trainable weight via `self.add_weight(trainable=False)`; weight tying is done by calling `tf.matmul(x, self.token_emb.embeddings, transpose_b=True)`.

In [ ]:
#  MiniLM - decoder-only transformer language model
class MiniLM(nn.Module):
    """
    Decoder-only transformer language model.
    Architecture: token_emb -> sinusoidal_PE -> n_layers x TransformerBlock -> lm_head
    """

    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.token_emb = nn.Embedding(vocab_size, d_model)

        # Stack n_layers identical transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.norm_out = nn.LayerNorm(d_model, eps=1e-5)
        pe = sinusoidal_pe(max_seq, d_model)
        self.register_buffer('pe', pe)

    def forward(self, token_ids, return_attn=False):
        """
        token_ids: (batch, seq_len)  int tensor
        Returns logits: (batch, seq_len, vocab_size)
        """
        S = token_ids.shape[1]
        x = self.token_emb(token_ids) + self.pe[:S]   # (B, S, d_model)

        # Block attention to future positions (decoder-only, causal)
        causal_mask = torch.triu(torch.ones(S, S, dtype=torch.bool), diagonal=1).to(x.device)
        all_attn = []

        # Run each transformer block in turn, collecting per-layer attention weights
        for block in self.blocks:
            x, aw = block(x, mask=causal_mask)
            all_attn.append(aw)
        x = self.norm_out(x)

        # Weight tying: reuse token embedding matrix as output projection
        logits = x @ self.token_emb.weight.T
        if return_attn:
            return logits, all_attn
        return logits


torch.manual_seed(42)
model_demo = MiniLM(vocab_size=VOCAB_SIZE, d_model=D_WORK, n_heads=NUM_HEADS, d_ff=D_FF, n_layers=2)

# Run one forward pass to instantiate lazy buffers, without tracking gradients
with torch.no_grad():
    _ = model_demo(torch.tensor([TOKEN_IDS]))
n_params = sum(p.numel() for p in model_demo.parameters())
print(f'MiniLM -> {n_params:,} trainable parameters')
for name, p in model_demo.named_parameters():
    print(f'  {name:<40} {tuple(p.shape)}')

In [ ]:
#  Training data - (context, next_token) pairs
full_corpus = [
    "the cat sat on the mat",
    "the dog ran over the fence",
    "a big cat jumped over the fence",
    "a dog sat on the mat",
    "the cat jumped over the fence",
    "the big dog ran on the mat",
]

# Build every (context-prefix, next-token) pair by sliding a window across each sentence
TRAIN_PAIRS = []
for sentence in full_corpus:
    ids = encode(sentence)
    for end in range(1, len(ids)):
        TRAIN_PAIRS.append((ids[:end], ids[end]))

print(f'Training pairs: {len(TRAIN_PAIRS)}')
print('\nFirst 6 examples:')
for ctx, tgt in TRAIN_PAIRS[:6]:
    print(f'  {[IDX2WORD[i] for i in ctx]}  ->  "{IDX2WORD[tgt]}"')


Now the training loop: full-batch gradient descent, predicting each next token from the position before it.


> **PyTorch → Keras:** the explicit 4-step training loop — `optimizer.zero_grad()`, forward pass, `loss.backward()`, `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)`, `optimizer.step()` — built on `torch.optim.Adam(model.parameters(), lr=3e-3)` and `nn.CrossEntropyLoss()`. **Keras/TF equivalent:** the manual-loop version wraps a forward pass inside `with tf.GradientTape() as tape:`, then `grads = tape.gradient(loss, model.trainable_variables)`, `grads, _ = tf.clip_by_global_norm(grads, 1.0)`, `optimizer.apply_gradients(zip(grads, model.trainable_variables))` — built on `tf.keras.optimizers.Adam(learning_rate=3e-3)` and `tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)`; Keras also offers the much higher-level `model.compile(...)` + `model.fit(...)` API that hides this loop entirely, which PyTorch has no built-in equivalent for.

In [ ]:
#  Training loop
# Left-pad every context to the batch's longest sequence so they can be stacked
def pad_collate(pairs, pad_id=0):
    max_len = max(len(ctx) for ctx, _ in pairs)
    xs, ys = [], []
    for ctx, tgt in pairs:
        pad = [pad_id] * (max_len - len(ctx))
        xs.append(pad + ctx)
        ys.append(tgt)
    return torch.tensor(xs, dtype=torch.long), torch.tensor(ys, dtype=torch.long)


torch.manual_seed(42)
model = MiniLM(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

EPOCHS = 300
x_train, y_train = pad_collate(TRAIN_PAIRS)
loss_history, acc_history = [], []

# Standard 4-step training loop: forward pass, backward pass, clip gradients, step
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    logits = model(x_train)
    last_logits = logits[:, -1, :]
    loss = loss_fn(last_logits, y_train)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    # Periodically log loss/accuracy and print progress
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            preds = torch.argmax(last_logits, dim=-1)
            acc = (preds == y_train).float().mean().item()
        loss_history.append(loss.item())
        acc_history.append(acc)
        if (epoch + 1) % 50 == 0:
            print(f'Epoch {epoch+1:4d} | loss={loss.item():.4f} | acc={acc:.2%}')

print('\nTraining complete.')


### Per-Position Loss: One Sequence, Many Lessons

The training loop above makes one next-token target explicit per padded context row. A decoder-only transformer can also process an entire sequence in parallel: logits at position $i$ are compared with the actual token at position $i+1$.

The next cell exposes that alignment and prints one loss contribution per token position. This is the canonical token-level loss microscope used by later fine-tuning chapters.

In [ ]:
# Expose the causal-LM loss one token position at a time.
probe_tokens = torch.tensor([encode(full_corpus[0])], dtype=torch.long)
model.eval()
with torch.no_grad():
    probe_logits = model(probe_tokens)

position_logits = probe_logits[:, :-1, :]
position_targets = probe_tokens[:, 1:]
position_losses = F.cross_entropy(
    position_logits.transpose(1, 2),
    position_targets,
    reduction="none",
)[0]

print("Context -> actual next token -> loss contribution")
for position, token_loss in enumerate(position_losses):
    context = " ".join(IDX2WORD[index] for index in probe_tokens[0, : position + 1].tolist())
    target = IDX2WORD[position_targets[0, position].item()]
    print(f"{context!r:28s} -> {target!r:10s} -> {token_loss.item():.4f}")

assert position_losses.shape == (probe_tokens.shape[1] - 1,)
print(f"Mean per-position loss: {position_losses.mean().item():.4f}")

### One Backward Pass: Many Token Lessons, One Update

Training is not autoregressive generation. The causal mask preserves left-to-right semantics, but the model computes logits and losses for all eligible sequence positions in parallel. Those position losses are averaged into one scalar before one backward pass and one optimizer step.

```mermaid
flowchart TD
    T["Token sequence"] --> L["Parallel logits<br/>one vector per position"]
    Y["Targets shifted left<br/>actual next tokens"] --> P["Per-position losses"]
    L --> P
    P --> A["Mean batch loss"]
    A --> B["One backward pass"]
    B --> G["Gradients on trainable parameters"]
    G --> U["One optimizer step"]
    U --> W["Updated model weights"]
```

The next cell traces that cycle on a fresh MiniLM copy so the trained model used later remains unchanged.

In [ ]:
# Trace one complete update on an isolated MiniLM copy.
torch.manual_seed(7)
update_probe = MiniLM(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=2)
probe_optimizer = torch.optim.SGD(update_probe.parameters(), lr=1e-2)


def probe_position_losses(probe_model):
    logits = probe_model(probe_tokens)[:, :-1, :]
    return F.cross_entropy(
        logits.transpose(1, 2),
        position_targets,
        reduction="none",
    )[0]


probe_optimizer.zero_grad()
before_losses = probe_position_losses(update_probe)
before_mean = before_losses.mean()
before_weights = update_probe.token_emb.weight.detach().clone()

before_mean.backward()
gradient_norm = torch.sqrt(
    sum(
        parameter.grad.detach().pow(2).sum()
        for parameter in update_probe.parameters()
        if parameter.grad is not None
    )
)
probe_optimizer.step()

with torch.no_grad():
    after_losses = probe_position_losses(update_probe)
weight_delta = (update_probe.token_emb.weight - before_weights).norm()

print(f"Position losses before: {[round(value, 4) for value in before_losses.tolist()]}")
print(f"Mean loss before:       {before_mean.item():.4f}")
print(f"Global gradient norm:   {gradient_norm.item():.4f}")
print(f"Embedding update norm:  {weight_delta.item():.6f}")
print(f"Mean loss after:        {after_losses.mean().item():.4f}")

assert torch.isfinite(gradient_norm)
assert weight_delta > 0
print("PASS: many position losses produced one gradient field and one weight update.")

In [ ]:
#  Training loss & accuracy plot
epochs_logged = list(range(10, EPOCHS + 1, 10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left panel: training loss curve
ax = axes[0]
ax.plot(epochs_logged, loss_history, color='royalblue', lw=2)
ax.fill_between(epochs_logged, loss_history, alpha=0.15, color='royalblue')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss'); ax.set_title('Training Loss')

# Right panel: next-token prediction accuracy curve
ax2 = axes[1]
ax2.plot(epochs_logged, [a * 100 for a in acc_history], color='mediumseagreen', lw=2)
ax2.fill_between(epochs_logged, [a * 100 for a in acc_history], alpha=0.15, color='mediumseagreen')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Next-Token Prediction Accuracy')
ax2.set_ylim(0, 105); ax2.axhline(100, color='grey', ls='--', lw=0.8)

plt.suptitle('MiniLM Training Progress', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


![Autoregressive generation and KV cache: how the model produces one token at a time](images/autoregressive-generation-and-kv-cache.png)

---

## Part 9 - Inference: Autoregressive Token Generation

At inference time, a decoder-only model repeatedly answers one question:

> Given **only the tokens already visible on the left**, what token should come next?

### Same Position, Different Left Context

Compare two unfinished sentences:

```text
Working at a ...
Washing clothes at a ...
```

The prediction position is structurally the same, but its hidden state is not context-blind:

```text
"Working at a"                  "Washing clothes at a"
       │                                  │
       ▼                                  ▼
attend to Working + at             attend to Washing + clothes + at
       │                                  │
       ▼                                  ▼
employment/location features       cleaning/fabric/location features
       │                                  │
       ▼                                  ▼
likely: bank, office, company      likely: laundromat, dry cleaner
```

The causal mask does **not** stop contextualization. It stops future leakage. The last visible position can blend information from every earlier token, so changing the prefix changes the query/key matches, the values blended into its hidden state, and the next-token distribution.

This is the decoder-only counterpart to a masked encoder example. A bidirectional encoder can contextualize a `[MASK]` token using words on both sides; a causal decoder has no placeholder in the middle and cannot use words to the right. It predicts after the prefix.

> **Real-tokenizer note:** production models generate subword tokens, not necessarily whole words. `laundromat` may require more than one decoding step even though the conceptual example writes it as one word.

### The Autoregressive Loop

1. Feed the current token prefix into the model.
2. Read the logits at the last position.
3. Apply temperature and softmax.
4. Sample or choose the highest-probability next token.
5. Append that token and repeat until an end token or stopping rule.

The next cell visualizes one step at a time. It recomputes the full prefix for clarity; production implementations normally cache earlier decoder keys and values so they do not rebuild the entire history at every step.

> **PyTorch → Keras:** `model.eval()` switches the module into inference mode (disables dropout/batch-norm training behaviour, none of which is used here but is standard practice); `with torch.no_grad():` disables autograd tracking so the forward pass doesn't build a computation graph. **Keras/TF equivalent:** pass `training=False` to the model/layer call (e.g. `model(ids, training=False)`) instead of a separate `.eval()` method; TF has no `no_grad()` context because it only tracks gradients inside an explicit `tf.GradientTape`, so a plain forward call outside a tape is already "no-grad" by default.

In [ ]:
#  Inference - autoregressive token generation
def generate_next(context_words, temperature=1.0):
    """Single next-token generation step with probability bar chart."""
    model.eval()

    # Run the forward pass without tracking gradients (inference only)
    with torch.no_grad():
        ids = torch.tensor([encode(' '.join(context_words))], dtype=torch.long)
        logits_inf = model(ids)
        last = logits_inf[0, -1, :]
        probs = torch.softmax(last / max(temperature, 1e-6), dim=-1).numpy()

    # Indices of the top-8 highest-probability next tokens, highest first
    topk_idx = probs.argsort()[::-1][:8]
    top_words = [IDX2WORD[int(i)] for i in topk_idx]
    top_probs = probs[topk_idx]

    fig, ax = plt.subplots(figsize=(8, 3))

    # Plot horizontal bars for the top-8 next-token probabilities, gold for the top pick
    ax.barh(top_words[::-1], top_probs[::-1],
            color=['gold' if w == top_words[0] else 'steelblue' for w in top_words[::-1]])
    ax.set_xlabel('Probability')
    ax.set_title(f'Next token probabilities | context: {context_words}  T={temperature}')
    ax.set_xlim(0, 1.0)

    # Annotate each bar with its exact probability value
    for bar, prob in zip(ax.patches, top_probs[::-1]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{prob:.3f}', va='center', fontsize=9)
    plt.tight_layout(); plt.show()

    best = IDX2WORD[int(probs.argmax())]
    print(f'  Greedy prediction: "{best}"')
    return best


#  Demo: step-by-step generation
print('=== Autoregressive generation ===')
print()
context = ['the']

# Greedily extend the context one token at a time, re-running the model each step
for step in range(5):
    print(f'Step {step+1}: context = {context}')
    next_tok = generate_next(context, temperature=0.8)
    context.append(next_tok)
    print()

print(f'Generated sequence: {" ".join(context)}')

![Scaling from toy (d=3) to production (d=768): same architecture, different dimensions](images/toy-to-production-transformers.png)

### From toy to real - same mechanism, bigger numbers

Everything you've built used tiny dimensions so the vectors stayed readable. A production model is the **identical machinery** scaled up.


In [ ]:
#  Toy (this notebook) vs. a real model (GPT-2 / DistilGPT-2)
rows = [
    ('embedding dim  d_model', D_MODEL, D_WORK, 768),
    ('attention heads',        '?',     NUM_HEADS, 12),
    ('dim per head  d_head',   '?',     D_HEAD, 64),
    ('feed-forward hidden',    '?',     D_FF, 3072),
    ('transformer layers',     '?',     2, 12),
    ('vocabulary size',        VOCAB_SIZE, VOCAB_SIZE, 50257),
]
print(f'{"component":<24}{"viz":>8}{"toy model":>12}{"GPT-2":>10}')
print('  ' + '-' * 52)
for name, viz, toy, real in rows:
    print(f'  {name:<22} {str(viz):>8} {str(toy):>12} {str(real):>10}')
print()
print('Every component in GPT-2 is identical in kind to what you built.')
print('  -> Scale, not novelty, is what makes GPT-2 impressive.')


---

## Part 11 - Why W_V Is a Relevance Filter, Not a Passthrough

$W_V$ is a **task-specific extraction lens**. Each attention layer has a different job. $W_V$ lets each layer extract exactly the slice it needs from each token's information.


#### Predict first - do we even need W_V?

Suppose we deleted $W_V$ entirely and blended the **raw token embeddings** directly.

**Predict:** if a *grammar* attention layer wants to know "is this token part of an active event?", it cares mainly about Dynamism and Animacy. Without $W_V$, can it ignore Concreteness?


In [ ]:
#  Part 11: W_V as a relevance filter
embs_sentence = embedding_matrix[TOKEN_IDS].numpy()   # (6, 3)

# W_V_action: extract the ACTION channel (Animacy + Dynamism)
W_V_action = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]], dtype=np.float32)

# W_V_object: extract the OBJECT channel (Concreteness + Animacy)
W_V_object = np.array([[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]], dtype=np.float32)

V_action = embs_sentence @ W_V_action   # (6, 2)
V_object = embs_sentence @ W_V_object   # (6, 2)

# Get "cat"'s attention row, then blend each V-lens by those same weights
attn_row_cat = attn_w.detach().numpy()[TOKENS.index('cat')]
blend_action = (attn_row_cat[:, None] * V_action).sum(0)
blend_object = (attn_row_cat[:, None] * V_object).sum(0)

# Plot each V-lens's projected values with "cat"'s blended context highlighted
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
specs = [
    (axes[0], V_action, blend_action, 'W_V_action - ACTION lens\n(Animacy x Dynamism)', 'Animacy', 'Dynamism'),
    (axes[1], V_object, blend_object, 'W_V_object - OBJECT lens\n(Concreteness x Animacy)', 'Concreteness', 'Animacy'),
]
for ax, V, blend, title, xl, yl in specs:
    ax.scatter(V[:, 0], V[:, 1], s=80, c='lightgray', edgecolor='#888', zorder=3)
    for i, tok in enumerate(TOKENS):
        ax.annotate(f'[{i}]{tok}', (V[i, 0], V[i, 1] + 0.03), fontsize=8, ha='center', color='dimgray')
    ax.scatter(*blend, s=280, color='gold', edgecolor='#b8860b', zorder=5,
               label='"cat" blended context', linewidths=2)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(title, fontsize=10); ax.legend(fontsize=9)
plt.suptitle('W_V shapes WHAT part of each token enters the weighted blend.\n'
             'Same sentence, same attention weights - different W_V - different context vector.',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('Key insight:')
print('  Attention weights (W_Q/W_K path) answer WHO gets blended.')
print('  W_V answers WHAT each token contributes to that blend.')
print('  -> W_V is not a passthrough; it is a task-specific extraction lens.')


---

## Part 12 - The Causal Triangle and the Accumulation Tower

The causal mask determines *how much of the sentence each position gets to know about*. Position 0 sees only itself. Position 5 sees all six tokens.

> **By the time the last position exits the final transformer block, it has absorbed a chain of increasingly enriched representations from every earlier position.**


In [ ]:
#  The Causal Triangle - explicit lower-triangular structure
S = SEQ_LEN

# Lower-triangular mask -- 1 where a query token may attend to a key token
mask_vis = np.tril(np.ones((S, S), dtype=int))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left panel: which key positions each query may attend to
ax = axes[0]
sns.heatmap(mask_vis.astype(float), ax=ax, cmap='Blues', vmin=0, vmax=1,
            xticklabels=TOKENS, yticklabels=TOKENS, linewidths=1.0, cbar=False,
            annot=mask_vis, fmt='d', annot_kws={'size': 14, 'weight': 'bold'})
ax.set_title('Causal Mask\n1 = allowed to attend  |  0 = blocked')
ax.set_xlabel('Key token'); ax.set_ylabel('Query token')

# Right panel: how many prior + current tokens each position can see
ax2 = axes[1]
history_counts = np.arange(1, S + 1)
bar_colors = plt.cm.Blues(np.linspace(0.35, 0.9, S))
ax2.bar(range(S), history_counts, color=bar_colors, edgecolor='white', lw=1)
ax2.set_xticks(range(S)); ax2.set_xticklabels(TOKENS)
ax2.set_ylabel('Tokens visible to this position')
ax2.set_title('How many tokens each position knows about')

# Label each bar with its exact visible-token count
for i, c in enumerate(history_counts):
    ax2.text(i, c + 0.05, str(c), ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Causal Triangle: position 0 is isolated; position 5 absorbs all 6 tokens',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


#### Predict first - does the accumulated history actually matter?

- **Full** context: `"the cat sat on the mat"` -> last token `"mat"`, 6 tokens of history
- **Truncated** context: `"the cat sat"` -> last token `"sat"`, 3 tokens of history

**Predict:** at block 3, will these two last-position vectors be very similar or clearly different?


> **PyTorch → Keras:** builds a fresh `MiniLM` with `torch.manual_seed(1)`, then traces intermediate layer outputs with `torch.no_grad()` and `torch.triu(torch.ones(...))` for the causal mask, reading each block's last-position vector via `.detach().numpy()`. **Keras/TF equivalent:** `tf.random.set_seed(1)`, calling the model with `training=False` in place of `no_grad()`, and `1 - tf.linalg.band_part(tf.ones((S_t, S_t)), -1, 0)` for the causal mask — the pattern of manually looping over sub-layers to capture intermediate activations works the same in a Keras subclassed model.

In [ ]:
#  Accumulation Tower: last-position richness grows with depth
torch.manual_seed(1)
tower_model = MiniLM(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=3)

# Run one forward pass to instantiate lazy state, without tracking gradients
with torch.no_grad():
    _ = tower_model(torch.tensor([TOKEN_IDS]))


def trace_last_position(ids_list):
    """Return the last-position hidden state after each transformer block."""
    tower_model.eval()
    with torch.no_grad():
        token_ids_t = torch.tensor([ids_list], dtype=torch.long)
        S_t = len(ids_list)
        x = tower_model.token_emb(token_ids_t) + tower_model.pe[:S_t]
        causal_m = torch.triu(torch.ones(S_t, S_t, dtype=torch.bool), diagonal=1)
        reps = []

        # Run through each block in turn, recording the last-position vector after each one
        for block in tower_model.blocks:
            x, _ = block(x, mask=causal_m)
            reps.append(x[0, -1, :].detach().numpy())
    return reps


reps_full  = trace_last_position(TOKEN_IDS)
reps_trunc = trace_last_position(TOKEN_IDS[:3])


# Cosine similarity between two vectors
def cos_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-9)
    b = b / (np.linalg.norm(b) + 1e-9)
    return float(a @ b)


# Compare full-context vs. truncated-context last-position representations at each block depth
similarities = [cos_sim(reps_full[d], reps_trunc[d]) for d in range(3)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]

# Left panel: representation norm growth per block, full vs truncated context
for label, reps, col in [('full (6 tokens)', reps_full, 'royalblue'), ('truncated (3 tokens)', reps_trunc, 'tomato')]:
    norms = [np.linalg.norm(r) for r in reps]
    ax.plot(range(1, 4), norms, 'o-', lw=2, label=label, color=col)
ax.set_xticks([1, 2, 3]); ax.set_xticklabels(['Block 1', 'Block 2', 'Block 3'])
ax.set_ylabel('L2 norm of last-position vector'); ax.set_title('Representation grows as context accumulates'); ax.legend(fontsize=9)

ax2 = axes[1]

# Colour-code bars by similarity tier (still similar / diverging / very different)
bar_c = ['#5ab4ac' if s > 0.9 else ('#d8b365' if s > 0.7 else 'tomato') for s in similarities]
ax2.bar(range(3), similarities, color=bar_c, alpha=0.9, edgecolor='white', lw=1.2)
ax2.plot(range(3), similarities, 'ko--', lw=1.5, ms=7)

# Label each bar with its exact similarity value
for i, s in enumerate(similarities):
    ax2.text(i, s + 0.01, f'{s:.3f}', ha='center', fontsize=11, fontweight='bold')
ax2.set_xticks([0, 1, 2]); ax2.set_xticklabels(['Block 1', 'Block 2', 'Block 3'])
ax2.set_ylabel('Cosine similarity')
ax2.set_title('Last-position vectors (full vs truncated context)\ndiverge as depth grows', fontsize=10)
ax2.set_ylim(0, 1.1); ax2.axhline(1.0, color='lightgray', ls='--', lw=1)

plt.suptitle('Accumulation Tower: same token ID, different history -> representations diverge with depth',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

for d, s in enumerate(similarities):
    tag = 'still similar' if s > 0.9 else ('diverging' if s > 0.7 else 'very different')
    print(f'  Block {d+1}: cosine similarity = {s:.3f}  ({tag})')
print('  -> Deeper stacks make the last position a richer accumulation point.')

---

## Part 14 - Cracking Open distilgpt2

Now we scale from our toy model to a real one: **DistilGPT-2** with:

- 6 transformer blocks
- 12 attention heads per block
- d_model = 768
- ~82M parameters

We will:
1. Load the model and inspect its architecture
2. Run a forward pass with `output_attentions=True` to capture all 6 layers x 12 heads
3. Visualise the attention patterns
4. Plot the next-token probability distribution


> **PyTorch → Keras:** `from transformers import GPT2LMHeadModel, GPT2Tokenizer` loads HuggingFace's **PyTorch-backend** DistilGPT-2 classes via `.from_pretrained('distilgpt2')`, then `gpt2.eval()` and inspecting `gpt2.transformer.wpe.weight.shape` / `.named_children()` walk the module tree. **Keras/TF equivalent:** `from transformers import TFGPT2LMHeadModel, GPT2Tokenizer` with the same `.from_pretrained('distilgpt2')` call gives the identical architecture on a Keras/TF backend; inspect sub-layers via `gpt2.transformer.wpe` and `gpt2.summary()`/`.layers` instead of `.named_children()` — HuggingFace maintains parallel `TF*` model classes specifically so the same checkpoint loads into either framework.

In [ ]:
#  Load DistilGPT-2 (PyTorch backend)
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt_tokenizer = GPT2Tokenizer.from_pretrained('distilgpt2')
gpt2 = GPT2LMHeadModel.from_pretrained('distilgpt2')
gpt2.eval()

cfg = gpt2.config
print('=== DistilGPT-2 Architecture ===')
print(f'  n_layer         : {cfg.n_layer}')
print(f'  n_head          : {cfg.n_head}')
print(f'  n_embd (d_model): {cfg.n_embd}')
print(f'  vocab_size      : {cfg.vocab_size}')
print(f'  n_positions     : {cfg.n_positions}  (max context)')
print(f'  wpe (pos. emb.) : {tuple(gpt2.transformer.wpe.weight.shape)}  (learned, not sinusoidal!)')
print()
n_params_gpt = sum(p.numel() for p in gpt2.parameters())
print(f'  Total parameters: {n_params_gpt:,}  (~{n_params_gpt/1e6:.1f}M)')
print()
print('Top-level modules:')
for name, module in gpt2.named_children():
    print(f'  {name}: {module.__class__.__name__}')
print()
print('First transformer block sub-modules:')
block0 = gpt2.transformer.h[0]
for name, mod in block0.named_children():
    print(f'  h[0].{name}: {mod.__class__.__name__}')
print()
print('Real BPE tokenisation, e.g. on an unfamiliar word:')
for rare_word in ['internationalization', 'zzzflorptastic']:
    pieces = gpt_tokenizer.tokenize(rare_word)
    print(f'  {rare_word!r:<24} -> {len(pieces)} piece(s): {pieces}')
print('  -> Unlike our one-word-one-token toy scheme, BPE splits unfamiliar words into')
print('     smaller, previously-seen sub-word chunks, so the vocabulary never needs an')
print('     <UNK> token for a whole word it has not seen before.')


> **PyTorch → Keras:** `gpt_tokenizer(PROMPT, return_tensors='pt')` returns PyTorch tensors; `torch.no_grad()` wraps the forward pass; `gpt2(**inputs, output_attentions=True)` requests all per-layer attention matrices; `torch.softmax` + `torch.topk` extract the top predictions. **Keras/TF equivalent:** `gpt_tokenizer(PROMPT, return_tensors='tf')` returns TF tensors instead; drop `no_grad()` (unneeded outside a `GradientTape`); the same `output_attentions=True` kwarg works on `TFGPT2LMHeadModel`; use `tf.nn.softmax` and `tf.math.top_k` in place of the PyTorch calls.

In [ ]:
#  Run inference with all attentions captured
PROMPT = 'The cat sat on the'
inputs = gpt_tokenizer(PROMPT, return_tensors='pt')
input_ids = inputs['input_ids']

gpt_tokens = [gpt_tokenizer.decode([int(t)]) for t in input_ids[0]]
print(f'Prompt       : {PROMPT!r}')
print(f'GPT-2 tokens : {gpt_tokens}')
print(f'Token IDs    : {input_ids[0].tolist()}')
print()

# Forward pass with all per-layer attention matrices captured
with torch.no_grad():
    out = gpt2(**inputs, output_attentions=True)

print(f'Number of attention layers returned: {len(out.attentions)}')
print(f'Each layer shape: {tuple(out.attentions[0].shape)}')
print(f'  (batch=1, n_heads=12, seq={len(gpt_tokens)}, seq={len(gpt_tokens)})')
print()

next_logits = out.logits[0, -1, :]
next_probs = torch.softmax(next_logits, dim=-1)
top10 = torch.topk(next_probs, k=10)

print(f'Top-10 next token predictions after "{PROMPT}":')
for rank, (tid, prob) in enumerate(zip(top10.indices.tolist(), top10.values.tolist()), 1):
    tok = gpt_tokenizer.decode([int(tid)])
    print(f'  {rank:2d}. {tok!r:<20} {float(prob):.4f}')


> **PyTorch → Keras:** `layer_attn[0].mean(dim=0).detach().numpy()` averages one layer's attention over the 12 heads for the heatmap grid. **Keras/TF equivalent:** `tf.reduce_mean(layer_attn[0], axis=0).numpy()` — same head-averaging reduction seen throughout this notebook, `axis=` in place of `dim=`.

In [ ]:
#  Attention pattern visualisation - all 6 layers, mean over heads
n_layers_gpt = len(out.attentions)
seq_len_gpt = len(gpt_tokens)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

# Plot each of the 6 layers' mean (over heads) attention pattern in its own panel
for layer_idx, (layer_attn, ax) in enumerate(zip(out.attentions, axes)):
    mean_attn = layer_attn[0].mean(dim=0).detach().numpy()
    sns.heatmap(mean_attn, ax=ax, cmap='viridis',
                xticklabels=gpt_tokens, yticklabels=gpt_tokens,
                linewidths=0.3, cbar_kws={'label': 'attn weight'})
    ax.set_title(f'Layer {layer_idx}  (mean over 12 heads)', fontsize=9)
    ax.set_xlabel('Key'); ax.set_ylabel('Query')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('DistilGPT-2: mean attention patterns across all 6 layers',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


> **PyTorch → Keras:** `out.attentions[0][0].detach().numpy()` indexes into the returned attention-tuple to pull out layer 0's per-head matrices as numpy for plotting. **Keras/TF equivalent:** `out.attentions[0][0].numpy()` — `TFGPT2LMHeadModel`'s `output_attentions=True` returns the same nested (layer, batch, head, seq, seq) structure, just without a `.detach()` step.

In [ ]:
#  Head diversity - compare individual heads in layer 0
layer0_attn = out.attentions[0][0].detach().numpy()   # (12, seq, seq)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Plot the first 4 of layer 0's 12 heads to show how differently each one attends
for h, ax in enumerate(axes):
    sns.heatmap(layer0_attn[h], ax=ax, cmap='Blues',
                xticklabels=gpt_tokens, yticklabels=gpt_tokens,
                linewidths=0.3, annot=True, fmt='.2f', annot_kws={'size': 8})
    ax.set_title(f'Layer 0  Head {h}', fontsize=9)
    ax.set_xlabel('Key'); ax.set_ylabel('Query')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)

plt.suptitle('DistilGPT-2 Layer 0: first 4 heads show diverse specialisation',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


> **PyTorch → Keras:** `torch.topk(next_probs, k=top_n)` returns the top-`k` probabilities and indices for the bar chart. **Keras/TF equivalent:** `tf.math.top_k(next_probs, k=top_n)` — same two-return-value `(values, indices)` API shape as `torch.topk`.

In [ ]:
#  Next-token probability bar chart
top_n = 15
topk_gpt = torch.topk(next_probs, k=top_n)
top_tokens_gpt = [gpt_tokenizer.decode([int(t)]) for t in topk_gpt.indices.tolist()]
top_probs_gpt = topk_gpt.values.detach().numpy()

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['gold'] + ['steelblue'] * (top_n - 1)

# Plot horizontal bars for the top-N next-token probabilities, gold for the top prediction
ax.barh(top_tokens_gpt[::-1], top_probs_gpt[::-1], color=colors[::-1], alpha=0.85)
ax.set_xlabel('Probability')
ax.set_title(f'DistilGPT-2 next-token probabilities\nPrompt: "{PROMPT}"',
             fontsize=11)

# Annotate each bar with its exact probability value
for bar, prob in zip(ax.patches, top_probs_gpt[::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{prob:.4f}', va='center', fontsize=8)
plt.tight_layout(); plt.show()

print(f'Top prediction: {top_tokens_gpt[0]!r}  (prob={top_probs_gpt[0]:.4f})')

> **PyTorch → Keras:** the manual sampling loop uses `torch.no_grad()`, `torch.topk` for top-k filtering, `torch.full_like(...)` + `.scatter_(0, top_idx, top_vals)` to build a filtered logits tensor, `torch.softmax`, `torch.multinomial` to sample one token, and `torch.cat` to grow the sequence. **Keras/TF equivalent:** drop `no_grad()`; `tf.math.top_k` for filtering; `tf.tensor_scatter_nd_update(tf.fill(logits_step.shape, float('-inf')), indices, top_vals)` in place of the in-place `scatter_` (TF tensors are immutable); `tf.nn.softmax`; `tf.random.categorical(logits_step[None, :], num_samples=1)` in place of `torch.multinomial`; `tf.concat([generated, next_id], axis=1)` in place of `torch.cat`. Note both HuggingFace backends also expose a much higher-level `model.generate(..., do_sample=True, temperature=..., top_k=...)` that hides this entire loop — this cell reimplements it manually to show the mechanics.

In [ ]:
#  Full autoregressive generation with distilgpt2


def gpt2_generate(prompt: str, max_new_tokens: int = 15, temperature: float = 0.8, top_k: int = 50):
    """Generate tokens one at a time, printing each step."""
    gpt2.eval()
    ids = gpt_tokenizer.encode(prompt, return_tensors='pt')
    print(f'Prompt: {prompt!r}')
    print(f'Starting IDs: {ids[0].tolist()}')
    print()

    generated = ids

    # Autoregressively sample one token at a time, feeding the growing sequence back in
    for step in range(max_new_tokens):
        with torch.no_grad():
            out_step = gpt2(generated)
        logits_step = out_step.logits[0, -1, :].clone()

        # Temperature + top-k filtering
        logits_step = logits_step / max(temperature, 1e-8)
        if top_k > 0:
            top_vals, top_idx = torch.topk(logits_step, k=top_k)
            filtered = torch.full_like(logits_step, float('-inf'))
            filtered.scatter_(0, top_idx, top_vals)
            logits_step = filtered

        probs_step = torch.softmax(logits_step, dim=-1)
        next_id = int(torch.multinomial(probs_step, num_samples=1).item())
        next_tok = gpt_tokenizer.decode([next_id])

        print(f'  Step {step+1:2d} - token ID {next_id:5d}  {next_tok!r:<20}  '
              f'p={probs_step[next_id].item():.4f}')

        generated = torch.cat([generated, torch.tensor([[next_id]])], dim=1)

        if next_id == gpt_tokenizer.eos_token_id:
            print('  [EOS - stopping]')
            break

    final_text = gpt_tokenizer.decode(generated[0].tolist(), skip_special_tokens=True)
    print()
    print(f'Final: {final_text!r}')
    return final_text


gpt2_generate('The cat sat on the', max_new_tokens=12, temperature=0.7)


---

## Part 2 Checkpoint: One Stack Can Read and Write

A decoder-only model places prompt and completion on one growing tape. The causal mask prevents future leakage; next-token loss supplies many lessons per sequence; autoregressive generation appends one prediction at a time.

[Continue to Part 3](03-encoder-decoder-and-cross-attention.ipynb) to see when a separate bidirectional source map and cross-attention are worth the additional structure.
